# Model C final: tuning hyperparameter (soal 3a)

**Model terpilih: `C2_finbert_art_cls_HGB`** (FinBERT, judul+lead per artikel, pooling `[CLS]`, rata-rata
per hari, PCA 20, digabung dengan 11 fitur harga, lalu HGB). Dipilih karena F1-macro validation-nya tertinggi
di antara semua varian C. F1-macro adalah metrik yang sama yang dipakai `tune()` untuk semua varian.

**Apa yang dilakukan di sini.** Fitur dibuat ulang persis seperti di `05b`, lalu HGB dicoba dengan grid yang
lebih besar (432 setelan, sebelumnya 16). Setelan terbaik dipilih berdasarkan **F1-macro validation**.
Test tidak disentuh sampai bagian terakhir, dan hanya dijalankan **sekali**.

| Hyperparameter | Nilai yang dicoba |
|---|---|
| `max_depth` | 2, 3, 4, 6 |
| `learning_rate` | 0.01, 0.03, 0.1 |
| `max_iter` | 100, 300, 600 |
| `min_samples_leaf` | 10, 20, 50, 100 |
| `l2_regularization` | 0, 1, 10 |

Jalankan dari folder `src/`. Embedding FinBERT dimuat dari cache `data/features_c/c2_finbert_art_cls.npy`
(dibuat di `05b`).

In [21]:
from c_common import *
import c_common, time

D, text = load_all()                      # corpus B_2, sama dengan 05b
print({s: len(d) for s, d in D.items()}, text.shape)

{'train': 825, 'val': 176, 'test': 178} (1202, 5)


## 1. Fitur (sama persis dengan `05b`)

Hari yang punya berita = hari dengan judul/lead tidak kosong. PCA di-fit hanya pada hari train yang punya
berita, lalu diterapkan ke semua hari; hari tanpa berita diberi 0 dan penanda `has`.

In [22]:
from sklearn.decomposition import PCA

TITLE, LEAD = "judul_gabungan", "lead_paragraph_gabungan"
day_text = (text[TITLE].fillna("") + ". " + text[LEAD].fillna("")).str.strip(" .")
has_news = (day_text.str.len() > 0).values
emb = np.load(FEATURES_C / "c2_finbert_art_cls.npy")

N_COMP, PREFIX = 20, "c2_art_cls"
feats = pd.DataFrame(emb, index=text.index)
tr = D["train"]["date"]
tr = tr[has_news[text.index.get_indexer(tr)]]
pca = PCA(n_components=N_COMP, random_state=SEED).fit(feats.loc[tr].values)
z = pca.transform(feats.values)
z[~has_news] = 0.0
comp = pd.DataFrame(z, index=text.index, columns=[f"{PREFIX}_{i}" for i in range(N_COMP)])
comp[f"{PREFIX}_has"] = has_news.astype(float)
for s in D:
    D[s] = D[s].merge(comp, left_on="date", right_index=True, how="left")
COLS = PRICE + list(comp.columns)
print("fitur:", len(COLS), "| variansi dijelaskan PCA:", round(pca.explained_variance_ratio_.sum(), 3))

fitur: 32 | variansi dijelaskan PCA: 0.792


## 2. Cek: hasil lama bisa direproduksi

Dengan setelan terbaik dari `05b`, skor validation harus sama dengan yang tersimpan di
`results_c/C2_finbert_art_cls_HGB.json`. Kalau sama, fiturnya memang identik.

In [23]:
old = json.loads((RESULTS_C / "C2_finbert_art_cls_HGB.json").read_text())
prob = make_model("HGB", old["best_params"]).fit(D["train"][COLS], D["train"]["target"]).predict_proba(D["val"][COLS])[:, 1]
sc = metrics(D["val"]["target"], (prob >= 0.5).astype(int), prob)
print("setelan lama:", old["best_params"])
print("tersimpan :", {k: round(v, 3) for k, v in old["val"].items()})
print("diulang   :", {k: round(v, 3) for k, v in sc.items()})
assert abs(sc["F1_macro"] - old["val"]["F1_macro"]) < 1e-9, "fitur tidak identik dengan 05b" 

setelan lama: {'max_depth': 3, 'learning_rate': 0.03, 'max_iter': 100, 'min_samples_leaf': 50}
tersimpan : {'DA': 0.591, 'BalAcc': 0.588, 'F1_macro': 0.588, 'MCC': 0.179, 'AUC': 0.568}
diulang   : {'DA': 0.591, 'BalAcc': 0.588, 'F1_macro': 0.588, 'MCC': 0.179, 'AUC': 0.568}


## 3. Grid search (validation saja)

Tiap setelan: latih di train, nilai di validation. Dipilih F1-macro tertinggi (kalau seri, setelan yang
muncul lebih dulu di grid, sama seperti `tune()`).

In [24]:
GRID_BIG = [dict(max_depth=d, learning_rate=l, max_iter=i, min_samples_leaf=m, l2_regularization=r)
            for d, l, i, m, r in product([2, 3, 4, 6], [0.01, 0.03, 0.1], [100, 300, 600],
                                         [10, 20, 50, 100], [0.0, 1.0, 10.0])]
print(len(GRID_BIG), "setelan")

rows, t0 = [], time.time()
for p in GRID_BIG:
    prob = make_model("HGB", p).fit(D["train"][COLS], D["train"]["target"]).predict_proba(D["val"][COLS])[:, 1]
    rows.append({**p, **metrics(D["val"]["target"], (prob >= 0.5).astype(int), prob)})
grid = pd.DataFrame(rows)
print(f"selesai dalam {time.time() - t0:.0f} dtk")

best_i = int(grid["F1_macro"].values.argmax())        # argmax = indeks pertama kalau seri
best_p = {k: grid.loc[best_i, k].item() for k in GRID_BIG[0]}
display(grid.sort_values("F1_macro", ascending=False).head(10).round(3))

432 setelan
selesai dalam 227 dtk


,max_depth,learning_rate,max_iter,min_samples_leaf,l2_regularization,DA,BalAcc,F1_macro,MCC,AUC
151,3,0.03,100,50,1.0,0.591,0.588,0.588,0.179,0.568
127,3,0.01,300,50,1.0,0.585,0.583,0.582,0.167,0.572
330,6,0.01,100,50,0.0,0.580,0.578,0.578,0.157,0.567
126,3,0.01,300,50,0.0,0.580,0.578,0.577,0.156,0.568
133,3,0.01,600,10,1.0,0.591,0.585,0.575,0.182,0.575
197,3,0.10,300,20,10.0,0.580,0.576,0.574,0.156,0.568
181,3,0.10,100,10,1.0,0.585,0.580,0.574,0.168,0.573
10,2,0.01,100,100,1.0,0.574,0.574,0.574,0.147,0.572
9,2,0.01,100,100,0.0,0.574,0.574,0.574,0.147,0.569
18,2,0.01,300,50,0.0,0.574,0.572,0.572,0.145,0.543


## 4. Sebelum vs sesudah tuning

Sebaran F1-macro di seluruh grid ditampilkan juga. Dengan hanya 176 hari validation, setelan terbaik dari
432 setelan sebagian bisa karena kebetulan; median grid memberi gambaran skor "biasa" untuk model ini.

In [29]:
sc_best = grid.loc[best_i, ["DA", "BalAcc", "F1_macro", "MCC", "AUC"]].astype(float).to_dict()
cmp = pd.DataFrame({"sebelum (16 setelan)": old["val"], f"sesudah ({len(GRID_BIG)} setelan)": sc_best}).T.round(3)
display(cmp)
print("setelan terbaik:", best_p)
print("F1-macro di seluruh grid:", grid["F1_macro"].describe()[["min", "25%", "50%", "75%", "max"]].round(3).to_dict())

,DA,BalAcc,F1_macro,MCC,AUC
sebelum (16 setelan),0.591,0.588,0.588,0.179,0.568
sesudah (432 setelan),0.591,0.588,0.588,0.179,0.568


setelan terbaik: {'max_depth': 3, 'learning_rate': 0.03, 'max_iter': 100, 'min_samples_leaf': 50, 'l2_regularization': 1.0}
F1-macro di seluruh grid: {'min': 0.462, '25%': 0.507, '50%': 0.522, '75%': 0.539, 'max': 0.588}


## 5. Simpan hasil tuning

Hasil disimpan ke `results_c/C2_finbert_art_cls_HGB.json`, menggantikan hasil tuning awal (16 setelan), dengan
`n_configs_tried` = jumlah setelan grid. Setelan terbaik dicek sama dengan setelan awal; kalau berbeda, sel ini
berhenti karena skor test yang tersimpan tidak lagi berlaku untuk model tersebut.

In [30]:
NAME = "C2_finbert_art_cls_HGB"
assert best_p == {**old["best_params"], "l2_regularization": 1.0}, "setelan terbaik berbeda dari setelan awal"
c_common._save(NAME, {"best_params": best_p, "val": sc_best, "n_configs_tried": len(GRID_BIG),
                      "val_auc_ci95": bootstrap_auc_ci(D["val"]["target"].values,
                          make_model("HGB", best_p).fit(D["train"][COLS], D["train"]["target"]).predict_proba(D["val"][COLS])[:, 1]),
                      "note": f"FinBERT art cls, PCA {N_COMP}; tuning akhir 3a ({len(GRID_BIG)} setelan HGB)"})
print(json.dumps(json.loads((RESULTS_C / f"{NAME}.json").read_text()), indent=1))

{
 "name": "C2_finbert_art_cls_HGB",
 "model": "HGB",
 "n_features": 32,
 "best_params": {
  "max_depth": 3,
  "learning_rate": 0.03,
  "max_iter": 100,
  "min_samples_leaf": 50,
  "l2_regularization": 1.0
 },
 "val": {
  "DA": 0.5909090909090909,
  "BalAcc": 0.5884938590820944,
  "F1_macro": 0.5875,
  "MCC": 0.17884712802707953,
  "AUC": 0.5681965093729799
 },
 "n_configs_tried": 432,
 "note": "FinBERT art cls, PCA 20; tuning akhir 3a (432 setelan HGB)",
 "test": {
  "DA": 0.5842696629213483,
  "BalAcc": 0.5610355987055016,
  "F1_macro": 0.5597593582887701,
  "MCC": 0.12700025714839938,
  "AUC": 0.5936569579288026
 },
 "test_auc_ci95": [
  0.515,
  0.662
 ],
 "val_auc_ci95": [
  0.504,
  0.654
 ]
}


## 6. Kontrol teks diacak

Fitur teks diacak antar hari (harga tetap), 30 kali. Kalau AUC asli tidak di atas kisaran ini, teks tidak
benar-benar membantu.

In [31]:
ctrl = shuffled_text_control("HGB", D, PRICE, list(comp.columns), best_p)
print("AUC val asli:", round(sc_best["AUC"], 3), "| teks diacak:", {k: round(v, 3) for k, v in ctrl.items()})

AUC val asli: 0.568 | teks diacak: {'auc_mean': 0.493, 'auc_min': 0.397, 'auc_max': 0.565}


## 7. Skor test

Setelan terbaik sama dengan setelan awal, jadi modelnya identik dengan yang sudah dites sekali di `06_ablation`
(`final_test`, prediksi per hari di `results_c/preds/C2_finbert_art_cls_HGB_test.csv`). Skor test dibaca dari
`results_c`, tidak dites ulang.

In [32]:
rec = json.loads((RESULTS_C / f"{NAME}.json").read_text())
print("test:", {k: round(v, 3) for k, v in rec["test"].items()}, "| AUC 95% CI", rec["test_auc_ci95"])

test: {'DA': 0.584, 'BalAcc': 0.561, 'F1_macro': 0.56, 'MCC': 0.127, 'AUC': 0.594} | AUC 95% CI [0.515, 0.662]
